# MiganCore SFT Identity Anchor Training

**Goal**: Bake identity into Qwen2.5-7B-Instruct via SFT + QLoRA (rank 32).

**Runtime**: Change to GPU (T4) before running.

**Time**: ~2-3 hours

**Cost**: FREE (Colab)

In [ ]:
# Cell 1: Verify GPU
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Install dependencies
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!pip install -q transformers datasets trl unsloth peft accelerate bitsandbytes sentencepiece protobuf scipy scikit-learn tqdm huggingface_hub

In [ ]:
# Cell 3: Download dataset from GitHub (or upload manually)
!wget -q https://raw.githubusercontent.com/tiranyx/migancore/main/data/workspace/identity_sft_200.jsonl -O identity_sft_200.jsonl
!wc -l identity_sft_200.jsonl

In [ ]:
# Cell 4: Download training script
!wget -q https://raw.githubusercontent.com/tiranyx/migancore/main/training/train_sft_identity.py -O train_sft_identity.py
!python -m py_compile train_sft_identity.py && echo 'Script OK'

In [ ]:
# Cell 5: Run Training
# NOTE: T4 has 16GB VRAM. Qwen2.5-7B 4-bit + rank 32 should fit.
# If OOM, reduce --lora-r to 16 or --max-seq-length to 1024.
!python train_sft_identity.py \
  --dataset identity_sft_200.jsonl \
  --output-dir ./identity_adapter \
  --base-model Qwen/Qwen2.5-7B-Instruct \
  --epochs 5 \
  --lora-r 32 \
  --batch-size 1 \
  --grad-accum 16 \
  --merge \
  --gguf

In [ ]:
# Cell 6: View eval report
import json
try:
    with open('eval_report.json') as f:
        report = json.load(f)
    print(json.dumps(report, indent=2))
except FileNotFoundError:
    print('No eval report found. Check training logs above.')

In [ ]:
# Cell 7: Download results
from google.colab import files
import os

# Zip results
!tar czf migan_identity_results.tar.gz identity_adapter/ merged_model/ *.gguf eval_report.json 2>/dev/null || echo 'Some files missing, zipping available ones'

if os.path.exists('migan_identity_results.tar.gz'):
    files.download('migan_identity_results.tar.gz')
else:
    print('No results to download')